# 06 · Logistic regression baseline

Stage 1 model: interpretable, fast, minimum reference point. Trained on v1 train, selected on v1 validation, reported on v1 test. Holdout stays unused.

In [ ]:
import pandas as pd

from cross_model_drift.data import load_split
from cross_model_drift.features import target_vector
from cross_model_drift.metrics import quality_metrics
from cross_model_drift.models import train_logistic
from cross_model_drift.notebook import setup_model_session
from cross_model_drift.tracking import clearml_task, log_metrics, log_parameters

nb = setup_model_session()
train = load_split("v1", "train", nb.config, engine=nb.engine)
valid = load_split("v1", "validation", nb.config, engine=nb.engine)
test = load_split("v1", "test", nb.config, engine=nb.engine)
len(train), len(valid), len(test)

In [ ]:
y_train = target_vector(train)
model = train_logistic(train, y_train, threshold=nb.threshold)
valid_metrics = quality_metrics(target_vector(valid), model.predict_proba(valid), threshold=nb.threshold)
test_metrics = quality_metrics(target_vector(test), model.predict_proba(test), threshold=nb.threshold)
pd.DataFrame([valid_metrics, test_metrics], index=["validation", "test"])

In [ ]:
path = model.save(nb.artifacts / "models" / "v1_logistic.joblib")
with clearml_task("train_logistic_baseline", config=nb.config, task_type="training", tags=["v1", "logistic"], init=False) as task:
    log_parameters(task, model.params)
    log_metrics(task, test_metrics, title="v1_test")
path

Set `init=True` after `~/.clearml/clearml.conf` points at the local server if you want this run logged.